# 1. Od transkrypcji do kontrolowalnych jednostek

[![Otwórz w Colabie](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caqdastm/ai_qda-workshop-1u/blob/main/04_vibe_coding/01_od_transkrypcji_do_jednostek.ipynb)

Cel: zaprojektować rejestr materiału, który zachowuje źródło, kolejność i dokładny tekst.

To jest ćwiczenie z **projektowania pipeline'u kodowania AI_QDA**.
Nie odtwarza autorskiego generatora pełnego wyniku. Kod techniczny jest
zwinięty; widoczne pozostają materiał, karta procedury, odpowiedzi modelu
i decyzja badacza.

W promptach **CZĘŚĆ BADAWCZA** pochodzi z karty uczestnika, a
**DODATEK TECHNICZNY** tylko dopasowuje jedną funkcję do notebooka.


## Rytm pracy

`cel kodowania → potrzebna procedura → karta → krótka kontrola →
porównanie dwóch polityk jednostki → inspekcja → decyzja badacza`

Nie projektujesz parsera DOCX. Oceniasz kontrakt jednostki, który
później pozwoli dostosować parser do innego korpusu.


## Dwa porządki pracy — nie mieszamy ich

**1. Tworzenie kodu:** krótką funkcję projektujesz w czacie AI
zintegrowanym z Colabem. Wysyłasz tam instrukcję o procedurze i
kontrakcie funkcji, a otrzymany kod wklejasz do wskazanej komórki.

**2. Analiza materiału:** dopiero działający notebook wysyła prompty i
ograniczony pakiet fragmentów przez API. W formularzu możesz wybrać
`gemini` albo `openai`; dalsze komórki pozostają takie same.

Dla wybranego providera dodaj w Colab Secrets tylko odpowiedni klucz:
`GEMINI_API_KEY` albo `OPENAI_API_KEY`. Przy OpenAI pole
`OPENAI_STORE` pozostaje jawną decyzją uczestnika; adapter zapisze jego
wartość i identyfikator odpowiedzi w lokalnym logu przebiegu.

Tryb `mock` sprawdza przepływ bez wysyłania danych. Czat Colaba służy do
vibe codingu, a `analysis_api` wyłącznie do porównań analitycznych.


In [ ]:
# @title Infrastruktura warsztatu — uruchom bez edycji { display-mode: "form" }
%pip install -q pandas "google-genai>=2.0.0" "openai>=2.0.0"

from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display

REPOSITORY_SLUG = "caqdastm/ai_qda-workshop-1u" # @param {type:"string"}
repo_folder = REPOSITORY_SLUG.replace("/", "__")
REPO_ROOT = Path("/content") / repo_folder
if not REPO_ROOT.exists():
    subprocess.run(
        ["git", "clone", "-q", f"https://github.com/{REPOSITORY_SLUG}.git", str(REPO_ROOT)],
        check=True,
    )
%cd $REPO_ROOT
support_dir = REPO_ROOT / "04_vibe_coding"
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from workshop_support import (
    AnalysisAPI,
    load_dataframe,
    load_workshop_packet,
    procedure_prompt,
    save_dataframe,
    save_json,
)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    WORKSPACE = Path("/content/drive/MyDrive/AI_QDA_Workshop")
except Exception:
    WORKSPACE = REPO_ROOT / "06_outputs" / "uczestnicy" / "AI_QDA_Workshop"
WORKSPACE.mkdir(parents=True, exist_ok=True)
print("Katalog przekazania między blokami:", WORKSPACE)


In [ ]:
# @title API analityczne — zmiana providera nie zmienia dalszych komórek { display-mode: "form" }
PROVIDER = "mock" # @param ["mock", "gemini", "openai"]
GEMINI_MODEL = "gemini-3.6-flash" # @param {type:"string"}
OPENAI_MODEL = "gpt-5.4-mini" # @param {type:"string"}
OPENAI_STORE = True # @param {type:"boolean"}
AUTHORIZE_API_CALLS = False # @param {type:"boolean"}
MAX_API_CALLS = 2 # @param {type:"integer"}

analysis_api = AnalysisAPI(
    provider=PROVIDER,
    gemini_model=GEMINI_MODEL,
    openai_model=OPENAI_MODEL,
    openai_store=OPENAI_STORE,
    authorize_api_calls=AUTHORIZE_API_CALLS,
    max_api_calls=MAX_API_CALLS,
)
print("Provider:", PROVIDER, "| model:", analysis_api.model, "| limit:", MAX_API_CALLS)


In [ ]:
# @title Materiał: przeczytaj sześć jednostek z rzeczywistych wywiadów PREWORK { display-mode: "form" }
packet = load_workshop_packet()
source_before = packet.copy(deep=True)
display(packet[["text_unit_id", "case_id", "sequence", "speaker", "text"]])


In [ ]:
# @title Karta procedury — edytuj język badawczy, nie kod { display-mode: "form" }
cel = "Zachować kontrolowalny ślad od jednostki kodowania do niezmienionego fragmentu źródłowego." # @param {type:"string"}
wejscie = "Rejestr fragmentów z ID, przypadkiem, kolejnością, mówcą, plikiem i tekstem." # @param {type:"string"}
rezultat = "Każdy wiersz ma stabilne ID i pozwala wrócić do dokładnego tekstu oraz jego miejsca w przypadku." # @param {type:"string"}
kontrola = "ID są niepuste i unikalne, kolejność jest jawna, tekst i liczba wierszy nie zmieniają się." # @param {type:"string"}
decyzja_badacza = "Czy tura, zdanie albo krótszy fragment jest właściwą jednostką dla danego modelu kodowania." # @param {type:"string"}

PROCEDURE_CARD = {
    "goal": cel,
    "input": wejscie,
    "observable_result": rezultat,
    "automatic_check": kontrola,
    "researcher_decision": decyzja_badacza,
}
display(pd.DataFrame([PROCEDURE_CARD]))


## Vibe coding w czacie Colaba: `check_unit_register`

1. Uruchom następną komórkę, aby wyświetlić instrukcję.
2. Otwórz panel czatu AI w Colabie i wklej całą instrukcję.
3. Poproś najpierw o krótkie powtórzenie kontraktu zwykłym językiem,
   a następnie o jedną funkcję — bez przebudowy notebooka.
4. Wklej otrzymaną funkcję do komórki **KOMÓRKA UCZESTNIKA**.

Na tym etapie nie korzystasz z klucza API i nie prosisz API
analitycznego o napisanie kodu.


In [ ]:
# @title Zbuduj prompt dla modelu piszącego jedną kontrolę { display-mode: "form" }
TECHNICAL_APPENDIX = """Napisz jedną funkcję check_unit_register(input_table, source_snapshot).
Zwróć tabelę checklisty z kolumnami kontrola i zaliczona.
Sprawdź: wymagane kolumny, niepuste i unikalne text_unit_id,
niezmienioną liczbę wierszy i tekst. Nie oceniaj trafności jednostki.
Nie zmieniaj tabeli wejściowej. Zwróć wyłącznie kod funkcji."""
FUNCTION_PROMPT = procedure_prompt(PROCEDURE_CARD, TECHNICAL_APPENDIX)
print(FUNCTION_PROMPT)


In [ ]:
# KOMÓRKA UCZESTNIKA: wklej pełną funkcję otrzymaną od modelu.
check_unit_register = None


In [ ]:
# @title Rozwiązanie awaryjne i czytelna checklista { display-mode: "form" }
def prepared_check_unit_register(input_table, source_snapshot):
    required = {"text_unit_id", "case_id", "source_file", "sequence", "speaker", "text"}
    ids = input_table["text_unit_id"].astype(str).str.strip() if "text_unit_id" in input_table else pd.Series(dtype=str)
    return pd.DataFrame([
        {"kontrola": "Są wymagane pola", "zaliczona": required.issubset(input_table.columns)},
        {"kontrola": "ID są niepuste", "zaliczona": len(ids) == len(input_table) and ids.ne("").all()},
        {"kontrola": "ID są unikalne", "zaliczona": ids.is_unique},
        {"kontrola": "Tekst zachowano", "zaliczona": input_table["text"].tolist() == source_snapshot["text"].tolist()},
        {"kontrola": "Źródło pozostało niezmienione", "zaliczona": input_table.equals(source_snapshot)},
    ])

if not callable(globals().get("check_unit_register")):
    check_unit_register = prepared_check_unit_register
checklist = check_unit_register(packet, source_before)
display(checklist)


## Analiza korpusu przez API

Teraz kod pomocniczy jest już w notebooku. Dwa kolejne wywołania API
dostają ten sam materiał, ale inaczej sformułowane zadania analityczne.
Porównujesz wpływ promptu, a nie SDK providera. Zmiana `gemini` na
`openai` odbywa się wyłącznie w formularzu **API analityczne**.


In [ ]:
# @title Dwa wywołania analityczne: porównaj polityki jednostki { display-mode: "form" }
shared = """Przeczytaj poniższy rejestr fragmentów. Nie koduj go.
Oceń konsekwencje proponowanej jednostki dla zachowania kontekstu,
precyzji przypisania i późniejszego porównywania kodów. Wskaż jedno
ryzyko oraz informację potrzebną badaczowi.\n\n""" + packet.to_csv(index=False)
response_a = analysis_api.run_analysis(
    shared + "\nPolityka A: każde zdanie jest osobną jednostką kodowania.",
    task_label="01_sentence_unit",
)
response_b = analysis_api.run_analysis(
    shared + "\nPolityka B: pełna tura jest kontekstem, a przypisanie wskazuje 1-3 kolejne zdania.",
    task_label="01_turn_context",
)
display(pd.DataFrame([
    {"wariant": "A — zdanie", "odpowiedź": response_a},
    {"wariant": "B — tura i fragment", "odpowiedź": response_b},
]))


## Powrót do materiału i decyzja badacza

Odpowiedzi API są kandydackie. Wróć do cytatów i zapisz własną decyzję
w formularzu poniżej. Checklista może wykryć błąd struktury, ale nie
potwierdza trafności kodu, kategorii ani granicy interpretacji.


In [ ]:
# @title Decyzja badacza { display-mode: "form" }
wybrana_polityka = "Tura jako kontekst, przypisanie do 1-3 kolejnych zdań" # @param {type:"string"}
uzasadnienie = "Pozwala zachować przebieg wypowiedzi, a jednocześnie wskazać dokładny fragment dowodowy." # @param {type:"string"}
czego_nie_rozstrzyga_kontrola = "Czy wybrana jednostka jest najlepsza dla konkretnego pytania badawczego." # @param {type:"string"}
RESEARCHER_DECISION = {
    "selected_policy": wybrana_polityka,
    "rationale": uzasadnienie,
    "interpretive_limit": czego_nie_rozstrzyga_kontrola,
}


In [ ]:
# @title Zapisz artefakty i przekazanie do bloku 2 { display-mode: "form" }
save_dataframe(WORKSPACE / "01_unit_register.csv", packet)
save_json(WORKSPACE / "01_procedure_card.json", PROCEDURE_CARD)
save_json(WORKSPACE / "01_researcher_decision.json", RESEARCHER_DECISION)
analysis_api.export_runs(WORKSPACE / "01_prompt_runs.jsonl")
print("Zapisano blok 1 w", WORKSPACE)


## Handoff

Blok 2 używa tego samego rejestru, ale podejmuje inną decyzję:
najpierw relewancja, potem 0-n procesualnych kodów D.
